In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# 1. Establish secure connection to your Dockerized database
DB_USER = os.getenv("DB_USER", "fintech_user")
DB_PASSWORD = os.getenv("DB_PASSWORD", "fintech_password")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "15432")
DB_NAME = os.getenv("DB_NAME", "fintech_db")
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

# 2. Extract normalized data via a relational SQL JOIN
query = """
    SELECT r.review_id, r.review_text, r.rating, r.review_date, 
           r.sentiment_label, r.sentiment_score, r.identified_theme, b.bank_name
    FROM reviews r
    JOIN banks b ON r.bank_id = b.bank_id;
"""
df = pd.read_sql(query, con=engine)
df['review_date'] = pd.to_datetime(df['review_date'])

# Ensure export directory exists
os.makedirs("../data/outputs", exist_ok=True)

# Set global professional plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 14})

# ==========================================================
# PLOT 1: Sentiment Distribution By Bank (Stacked Bar Chart)
# ==========================================================
plt.figure(figsize=(10, 6))
sentiment_counts = df.groupby(['bank_name', 'sentiment_label']).size().unstack(fill_value=0)
# Normalize to percentages for a clean comparative layout
sentiment_pct = sentiment_counts.div(sentiment_counts.sum(axis=1), axis=0) * 100

ax = sentiment_pct.plot(kind='bar', stacked=True, color=['#E63946', '#4EA8DE'], figsize=(10, 6), edgecolor='black')
plt.title("Comparative Sentiment Distribution Across Ethiopian Banks (%)")
plt.xlabel("Financial Institution")
plt.ylabel("Percentage Baseline (%)")
plt.xticks(rotation=0)
plt.legend(title="NLP Sentiment Label", loc="lower right")
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_x(), p.get_y() 
    if height > 0:
        ax.annotate(f'{height:.1f}%', (x + width/2, y + height/2), ha='center', va='center', color='white', weight='bold')

plt.tight_layout()
plt.savefig("../data/outputs/sentiment_distribution.png", dpi=300)
plt.close()

# ==========================================================
# PLOT 2: Rating Distribution Profile Per Bank (Boxplot)
# ==========================================================
plt.figure(figsize=(10, 6))
sns.boxplot(x='bank_name', y='rating', data=df, palette="Blues", width=0.5)
sns.stripplot(x='bank_name', y='rating', data=df, color="orange", alpha=0.3, jitter=0.2)
plt.title("Structural Distribution of User Star Ratings")
plt.xlabel("Financial Institution")
plt.ylabel("Star Rating (1-5 Constraints)")
plt.tight_layout()
plt.savefig("../data/outputs/rating_boxplot.png", dpi=300)
plt.close()

# ==========================================================
# PLOT 3: Theme Frequency Analysis Matrix (Horizontal Bar Charts)
# ==========================================================
fig, axes = plt.subplots(3, 1, figsize=(11, 14), sharex=True)
banks = ['CBE', 'BOA', 'Dashen']
colors = ['#1D3557', '#457B9D', '#A8DADC']

for i, bank in enumerate(banks):
    bank_df = df[df['bank_name'] == bank]
    theme_data = bank_df['identified_theme'].value_counts()
    
    sns.barplot(x=theme_data.values, y=theme_data.index, ax=axes[i], color=colors[i], edgecolor='black')
    axes[i].set_title(f"Operational Theme Densities: {bank}")
    axes[i].set_xlabel("Absolute Written Review Count")
    axes[i].set_ylabel("")

plt.tight_layout()
plt.savefig("../data/outputs/theme_frequencies.png", dpi=300)
plt.close()

# ==========================================================
# PLOT 4: Sentiment Trajectory Over Time (Rolling Sentiment Trend)
# ==========================================================
plt.figure(figsize=(12, 6))
df['is_positive'] = df['sentiment_label'].apply(lambda x: 1 if x == 'POSITIVE' else 0)

for bank in banks:
    bank_df = df[df['bank_name'] == bank].sort_values('review_date')
    # Calculate a 7-day rolling average of positive sentiment density
    rolling_sentiment = bank_df.set_index('review_date')['is_positive'].rolling('7D').mean() * 100
    plt.plot(rolling_sentiment.index, rolling_sentiment.values, label=f"{bank} (7D Rolling Average)", linewidth=2.5)

plt.title("Temporal Trajectory of Positive User Sentiment (Rolling 7-Day Window)")
plt.xlabel("Timeline Index")
plt.ylabel("Positive Sentiment Ratio (%)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("../data/outputs/sentiment_trends.png", dpi=300)
plt.close()

print("🎉 Visualizations compiled and saved smoothly into data/outputs/!")

/tmp/ipykernel_66357/4274309695.py:61: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='bank_name', y='rating', data=df, palette="Blues", width=0.5)


🎉 Visualizations compiled and saved smoothly into data/outputs/!


<Figure size 1000x600 with 0 Axes>